# Training a Simple Workshop Recommendation Model

This notebook trains a simple machine learning model to predict a student's preferred workshop format based on their background, confidence, interests, and learning barriers.

This is not meant to replace human judgment. It is a teaching example showing how data preparation, modeling, evaluation, and responsible reflection fit together.

In [0]:
df = spark.table("ai_access_lab_students")
pdf = df.toPandas()

pdf.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

features = [
    "year",
    "major",
    "prior_coding_experience",
    "ai_exposure",
    "main_learning_barrier",
    "topic_interest",
    "confidence_ai_ml",
    "has_attended_tech_event",
    "confidence_group",
    "beginner_status"
]

target = "preferred_workshop_format"

X = pdf[features]
y = pdf[target]

categorical_features = [
    "year",
    "major",
    "prior_coding_experience",
    "ai_exposure",
    "main_learning_barrier",
    "topic_interest",
    "has_attended_tech_event",
    "confidence_group",
    "beginner_status"
]

numeric_features = ["confidence_ai_ml"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

clf.fit(X_train, y_train)

preds = clf.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print("Accuracy:", accuracy)
print()
print(classification_report(y_test, preds))

In [0]:
import mlflow
import mlflow.sklearn

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="ai_access_lab_workshop_recommender"):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    
    mlflow.log_metric("accuracy", accuracy)
    
    print("Logged model with accuracy:", accuracy)

In [0]:
import mlflow
import mlflow.sklearn

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="ai_access_lab_workshop_recommender"):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    
    mlflow.log_metric("accuracy", accuracy)
    
    print("Logged model with accuracy:", accuracy)

## Responsible AI Reflection

This model should not be used to decide what a student is capable of learning. It only predicts a preferred workshop format from synthetic data.

In a real campus setting, this kind of model would need:
- consent from students,
- careful privacy protection,
- transparency about how recommendations are made,
- human review,
- and a way for students to choose their own learning path.

The most important lesson is that AI systems are not just models. They depend on data quality, assumptions, context, evaluation, and human responsibility.